## UAS Audio Classification: Drone, Helicopter, Background

This project will be a three-class classification problem: drone, helicopter, background, where the helicopter class is included to test whether models can distinguish target drones from acoustically similar legitimate rotorcraft rather than dismissing them both into a single aircraft category.

We'll evalutae the generalization to new and unknown recording conditions, discrimination between drones and legitimate rotorcraft such as medical helicopters, tolerance of low signal-to-noise ratio (SNR).

In [2]:
import os
import numpy as np
import pandas as pd
import torch
import warnings

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

import config as cfg
import data
import models
import train

torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

os.makedirs("figures", exist_ok=True)
cfg.ensure_dirs()

Using device: cuda


In [3]:
warnings.filterwarnings("ignore")

pio.templates.default = "plotly_white"
pio.renderers.default = "notebook_connected+vscode"

# Global font size for all charts
pio.templates["plotly_white"].layout.font.size = 16
pio.templates["plotly_white"].layout.title.font.size = 20
pio.templates["plotly_white"].layout.xaxis.title.font.size = 16
pio.templates["plotly_white"].layout.yaxis.title.font.size = 16

### Data

- **Svanström** (primary): drone, helicopter, and background audio recorded at three Swedish airports
- **Al-Emadi** (cross-source holdout): drone and background audio from different microphones and environments
- **ESC-50**: environmental noise used to degrade SNR at test time

Splits are taken at the "recording level" before segmentation, so no two overlapping 2-second windows from the same file can land in different splits, preventing the form of data leakage.

In [4]:
svan_df = data.build_svanstrom_manifest()
aleem_df = data.build_al_emadi_manifest()
esc_df = data.build_esc50_manifest()

print(f"Svanström files: {len(svan_df)}")
print(svan_df["label"].value_counts())
print(f"Al-Emadi files (cross-source holdout): {len(aleem_df)}")
print(f"ESC-50 noise clips: {len(esc_df)}")

Svanström files: 90
label
background    30
drone         30
helicopter    30
Name: count, dtype: int64
Al-Emadi files (cross-source holdout): 11704
ESC-50 noise clips: 320


In [5]:
split_df = data.recording_level_split(svan_df)
train_manifest = split_df[split_df["split"] == "train"]
val_manifest = split_df[split_df["split"] == "val"]
test_manifest = split_df[split_df["split"] == "test"]

In [6]:
# One example spectrogram per class
fig = make_subplots(rows=1, cols=3, subplot_titles=cfg.CLASSES)

for col, cls in enumerate(cfg.CLASSES, start=1):
    path = svan_df[svan_df["label"] == cls]["path"].iloc[0]
    y = data.load_audio(path)
    seg = data.segment_audio(y)[0]
    spec = data.log_mel_spectrogram(seg)
    fig.add_trace(
        go.Heatmap(z=spec, colorscale="Viridis", showscale=(col == 3)),
        row=1, col=col,
    )

fig.update_layout(title="Log-mel spectrograms, 128 mel bands x ~200 frames", width=1400)
fig.update_xaxes(title_text="Frame")
fig.update_yaxes(title_text="Mel bin", col=1)
fig.write_image("figures/example_spectrograms.png", scale=3)
fig.show()

### Architecture 1: Linear SVM on hand-engineered features

MFCC statistics (mean and std across time) plus spectral centroid, roll-off, bandwidth, zero-crossing rate, and RMS energy. `CalibratedClassifierCV` wraps `LinearSVC` for probability estimates for per-class AUC and bootstrap confidence intervals. Class weight is set to `balanced` so that the linear boundary is not pulled by the dominant class.

This followed the Week 2 session on Support Vector Machines; when a linear classifier is often surprisingly competitive when the information needed for the task is captured by interpretable spectral features.

In [7]:
svm_pipeline = train.train_svm(train_manifest)

X_test_svm, y_test_svm = train.extract_svm_dataset(test_manifest)
svm_test = train.evaluate_svm(svm_pipeline, X_test_svm, y_test_svm)

print(f"SVM macro-F1: {svm_test['macro_f1']:.4f} "
      f"[{svm_test['f1_ci'][1]:.4f}, {svm_test['f1_ci'][2]:.4f}]")
print(f"Per-class AUC: {svm_test['auc_per_class']}")

SVM macro-F1: 0.8742 [0.8057, 0.9251]
Per-class AUC: {'drone': 0.9681481481481482, 'helicopter': 0.9792592592592592, 'background': 0.9496296296296296}


### Architecture 2: Five-block CNN trained from scratch

Five blocks of `Conv2d` -> `BatchNorm` -> `ReLU` -> `MaxPool` -> `Dropout` layers with increasing number of channels in each block (32, 64, 128, 256, 512) which are followed by an `Adaptive Average Pool` layer and a two-layer classifier. The model was trained using log-mel spectrogram data.

A sequence of three 3×3 convolutional layers increase the size of the receptive field through the depth of the network (Week 5). The use of batch normalization prevents the vanishing gradient problem during backpropagation (Week 4). Dropout (Week 4) along with L2 regularization of weights prevent overfitting due to the limited number of samples used to train the models.

In [8]:
train_ds = data.AudioDataset(train_manifest, augment=True)
val_ds = data.AudioDataset(val_manifest, augment=False)
test_ds = data.AudioDataset(test_manifest, augment=False)

train_loader = data.make_loader(train_ds, cfg.CNN_BATCH_SIZE, shuffle=True)
val_loader = data.make_loader(val_ds, cfg.CNN_BATCH_SIZE, shuffle=False)
test_loader = data.make_loader(test_ds, cfg.CNN_BATCH_SIZE, shuffle=False)

print(f"Train segments: {len(train_ds)}")
print(f"Val segments:   {len(val_ds)}")
print(f"Test segments:  {len(test_ds)}")

Train segments: 567
Val segments:   108
Test segments:  135


In [9]:
class_weights = train.compute_class_weights(train_manifest, device)
cnn = models.SimpleCNN().to(device)
total, trainable = models.count_params(cnn)
print(f"SimpleCNN parameters: {trainable:,} trainable / {total:,} total")

optimizer = torch.optim.Adam(cnn.parameters(), lr=cfg.CNN_LR,
                             weight_decay=cfg.CNN_WEIGHT_DECAY)

cnn, cnn_hist = train.train_model(
    cnn, train_loader, val_loader, optimizer, device,
    num_epochs=cfg.CNN_MAX_EPOCHS,
    patience=cfg.CNN_PATIENCE,
    class_weights=class_weights,
)

SimpleCNN parameters: 1,702,083 trainable / 1,702,083 total

Epoch 1/60


Train Loss: 0.9560
Val Loss: 0.9812, Val F1: 0.5813

Epoch 2/60


Train Loss: 0.8322
Val Loss: 1.5928, Val F1: 0.6675

Epoch 3/60


Train Loss: 0.7441
Val Loss: 2.6890, Val F1: 0.3915

Epoch 4/60


Train Loss: 0.6274
Val Loss: 0.9479, Val F1: 0.5052

Epoch 5/60


...

Train Loss: 0.5599
Val Loss: 0.3931, Val F1: 0.8031

Epoch 25/60


Train Loss: 0.3578
Val Loss: 0.5599, Val F1: 0.7452

Epoch 26/60


Train Loss: 0.3953
Val Loss: 0.2997, Val F1: 0.8918

Epoch 27/60


Train Loss: 0.4584
Val Loss: 0.6529, Val F1: 0.7243

Epoch 28/60


Train Loss: 0.4021
Val Loss: 0.5607, Val F1: 0.8063
Early stop at epoch 28 (best val_f1 0.9071)


In [10]:
hist_df = pd.DataFrame({
    "epoch": range(1, len(cnn_hist["train_loss"]) + 1),
    "train_loss": cnn_hist["train_loss"],
    "val_loss": cnn_hist["val_loss"],
    "val_f1": cnn_hist["val_f1"],
})

fig = make_subplots(rows=1, cols=2, subplot_titles=["Loss", "Validation macro-F1"])
fig.add_trace(go.Scatter(x=hist_df["epoch"], y=hist_df["train_loss"], name="train loss"), row=1, col=1)
fig.add_trace(go.Scatter(x=hist_df["epoch"], y=hist_df["val_loss"], name="val loss"), row=1, col=1)
fig.add_trace(go.Scatter(x=hist_df["epoch"], y=hist_df["val_f1"], name="val F1",
                         showlegend=False, line=dict(color="green")), row=1, col=2)
fig.update_layout(title="CNN training history", width=1500)
fig.update_xaxes(title_text="Epoch")
fig.write_image("figures/cnn_curves.png", scale=3)
fig.show()

In [11]:
cnn_test = train.full_evaluate(cnn, test_loader, device)
print(f"CNN macro-F1: {cnn_test['macro_f1']:.4f} "
      f"[{cnn_test['f1_ci'][1]:.4f}, {cnn_test['f1_ci'][2]:.4f}]")
print(f"Per-class AUC: {cnn_test['auc_per_class']}")

CNN macro-F1: 0.9095 [0.8572, 0.9533]
Per-class AUC: {'drone': 1.0, 'helicopter': 0.9923456790123457, 'background': 0.934074074074074}


### Architecture 3: PANNs CNN10 with progressive unfreezing

CNN10 from Kong et al. (2020), pretrained on AudioSet's roughly two million audio clips with the same conv-block topology as the scratch CNN above, the only change is whether the backbone starts from ImageNet-style random initialisation or from weights that have already learned general audio features. 

Three-stage fine-tuning:
1. **Head only** (lr 1e-3). Classifier adapts to our three-class label space while the backbone remains frozen.
2. **Head + last conv block** (head lr 1e-3, backbone 1e-5). Differential learning rates let the deepest layer adapt to domain-specific features while shallower layers are preserved.
3. **Head + last two conv blocks**. More task-specific adaptation.

In [12]:
ckpt_path = cfg.CKPT_ROOT / "Cnn10_mAP=0.380.pth"
models.download_pann_checkpoint(ckpt_path)

pann = models.load_pann_cnn10(ckpt_path).to(device)
total, _ = models.count_params(pann)
print(f"PANN CNN10 parameters: {total:,}")

Loaded 50 pretrained tensors from C:\University\Data Mining 2 AE3\artefact\checkpoints\Cnn10_mAP=0.380.pth
PANN CNN10 parameters: 4,950,339


In [13]:
pann_train_loader = data.make_loader(train_ds, cfg.PANN_BATCH_SIZE, shuffle=True)
pann_val_loader = data.make_loader(val_ds, cfg.PANN_BATCH_SIZE, shuffle=False)
pann_test_loader = data.make_loader(test_ds, cfg.PANN_BATCH_SIZE, shuffle=False)

pann, pann_hist = train.train_pann_progressive(
    pann, pann_train_loader, pann_val_loader, device,
    class_weights=class_weights,
)


PANN stage: head_only
Trainable: 264,195 / 4,950,339

Epoch 1/10


Train Loss: 1.0413
Val Loss: 0.9660, Val F1: 0.5612

Epoch 2/10


Train Loss: 0.8297
Val Loss: 0.7757, Val F1: 0.7666

Epoch 3/10


Train Loss: 0.6905
Val Loss: 0.6811, Val F1: 0.7967

Epoch 4/10


Train Loss: 0.6106
Val Loss: 0.6227, Val F1: 0.7148

Epoch 5/10


...

Train Loss: 0.4182
Val Loss: 0.2586, Val F1: 0.8795

Epoch 17/20


Train Loss: 0.2477
Val Loss: 0.2525, Val F1: 0.8606

Epoch 18/20


Train Loss: 0.2861
Val Loss: 0.2652, Val F1: 0.8699

Epoch 19/20


Train Loss: 0.3557
Val Loss: 0.2597, Val F1: 0.8887

Epoch 20/20


Train Loss: 0.4651
Val Loss: 0.2513, Val F1: 0.8608


In [14]:
# Put the three stages onto a single epoch axis
rows = []
offset = 0
boundaries = []

for stage_name, hist in pann_hist.items():
    for i, (tl, vl, vf) in enumerate(zip(hist["train_loss"],
                                          hist["val_loss"],
                                          hist["val_f1"])):
        rows.append({"epoch": offset + i + 1, "stage": stage_name,
                     "train_loss": tl, "val_loss": vl, "val_f1": vf})
    offset += len(hist["train_loss"])
    boundaries.append(offset)

pann_df = pd.DataFrame(rows)

fig = make_subplots(rows=1, cols=2, subplot_titles=["Loss", "Validation macro-F1"])
fig.add_trace(go.Scatter(x=pann_df["epoch"], y=pann_df["train_loss"], name="train loss"), row=1, col=1)
fig.add_trace(go.Scatter(x=pann_df["epoch"], y=pann_df["val_loss"], name="val loss"), row=1, col=1)
fig.add_trace(go.Scatter(x=pann_df["epoch"], y=pann_df["val_f1"], name="val F1",
                         showlegend=False, line=dict(color="green")), row=1, col=2)

for b in boundaries[:-1]:
    fig.add_vline(x=b + 0.5, line_dash="dash", line_color="grey", row=1, col=1)
    fig.add_vline(x=b + 0.5, line_dash="dash", line_color="grey", row=1, col=2)

fig.update_layout(title="PANN CNN10 three-stage progressive unfreezing", width=1500)
fig.update_xaxes(title_text="Epoch (cumulative across stages)")
fig.write_image("figures/pann_curves.png", scale=3)
fig.show()

In [15]:
pann_test = train.full_evaluate(pann, pann_test_loader, device)
print(f"PANN macro-F1: {pann_test['macro_f1']:.4f} "
      f"[{pann_test['f1_ci'][1]:.4f}, {pann_test['f1_ci'][2]:.4f}]")
print(f"Per-class AUC: {pann_test['auc_per_class']}")

PANN macro-F1: 0.9317 [0.8864, 0.9700]
Per-class AUC: {'drone': 0.9962962962962962, 'helicopter': 0.9918518518518519, 'background': 0.9520987654320988}


### Three evaluations

Three questions, three evaluations:

1. **In-distribution.** Which model wins on the held-out Svanström test set?
2. **Cross-source.** How far does each model's performance drop on Al-Emadi, data with different microphones, drones, and recording environments that was never seen during training?
3. **Noise robustness.** How does macro-F1 degrade as ESC-50 background noise is added at worse and worse SNRs (clean, +10, +5, 0, −5 dB)?

In [16]:
results = {
    "SVM":  {"in_dist": svm_test},
    "CNN":  {"in_dist": cnn_test},
    "PANN": {"in_dist": pann_test},
}

summary = []
for name, r in results.items():
    f1_mean, f1_lo, f1_hi = r["in_dist"]["f1_ci"]
    summary.append({
        "model": name,
        "macro_f1": r["in_dist"]["macro_f1"],
        "f1_lo": f1_lo,
        "f1_hi": f1_hi,
        "auc_drone": r["in_dist"]["auc_per_class"]["drone"],
        "auc_helicopter": r["in_dist"]["auc_per_class"]["helicopter"],
        "auc_background": r["in_dist"]["auc_per_class"]["background"],
    })

summary_df = pd.DataFrame(summary)
print(summary_df.round(4).to_string(index=False))

fig = px.bar(summary_df, x="model", y="macro_f1",
             error_y=summary_df["f1_hi"] - summary_df["macro_f1"],
             error_y_minus=summary_df["macro_f1"] - summary_df["f1_lo"],
             title="In-distribution macro-F1 with 95% bootstrap CI", width=1200)
fig.write_image("figures/in_distribution.png", scale=3)
fig.show()

model  macro_f1  f1_lo  f1_hi  auc_drone  auc_helicopter  auc_background
  SVM    0.8742 0.8057 0.9251     0.9681          0.9793          0.9496
  CNN    0.9095 0.8572 0.9533     1.0000          0.9923          0.9341
 PANN    0.9317 0.8864 0.9700     0.9963          0.9919          0.9521


In [17]:
fig = make_subplots(rows=1, cols=3, subplot_titles=list(results.keys()))

for col, (name, r) in enumerate(results.items(), start=1):
    cm = r["in_dist"]["confusion_matrix"]
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    fig.add_trace(
        go.Heatmap(z=cm_norm, x=cfg.CLASSES, y=cfg.CLASSES, colorscale="Blues", showscale=(col == 3),
                   text=cm, texttemplate="%{text}", zmin=0, zmax=1), row=1, col=col,
    )

fig.update_layout(title="Confusion matrices (row-normalised, counts shown)")
fig.write_image("figures/confusion_matrices.png", scale=3, width=1700, height=400)
fig.show()

In [18]:
results["SVM"]["cross"]  = train.cross_source_eval(svm_pipeline, aleem_df, device, is_torch=False)
results["CNN"]["cross"]  = train.cross_source_eval(cnn, aleem_df, device, is_torch=True, batch_size=cfg.CNN_BATCH_SIZE)
results["PANN"]["cross"] = train.cross_source_eval(pann, aleem_df, device, is_torch=True, batch_size=cfg.PANN_BATCH_SIZE)

cross_rows = []
for name, r in results.items():
    cross_rows.append({"model": name, "split": "in-dist", "macro_f1": r["in_dist"]["macro_f1"]})
    cross_rows.append({"model": name, "split": "cross-source", "macro_f1": r["cross"]["macro_f1"]})

cross_df = pd.DataFrame(cross_rows)
print(cross_df.round(4).to_string(index=False))

fig = px.bar(cross_df, x="model", y="macro_f1", color="split", barmode="group",
             title="In-distribution vs cross-source (Al-Emadi) macro-F1", width=1300)
fig.write_image("figures/cross_source.png", scale=3)
fig.show()

model        split  macro_f1
  SVM      in-dist    0.8742
  SVM cross-source    0.4258
  CNN      in-dist    0.9095
  CNN cross-source    0.5178
 PANN      in-dist    0.9317
 PANN cross-source    0.5681


In [19]:
snr_svm  = train.snr_degradation_curve(svm_pipeline, test_manifest, esc_df, device, is_torch=False)
snr_cnn  = train.snr_degradation_curve(cnn, test_manifest, esc_df, device, is_torch=True, batch_size=cfg.CNN_BATCH_SIZE)
snr_pann = train.snr_degradation_curve(pann, test_manifest, esc_df, device, is_torch=True, batch_size=cfg.PANN_BATCH_SIZE)

for name, df in [("SVM", snr_svm), ("CNN", snr_cnn), ("PANN", snr_pann)]:
    df["model"] = name

snr_all = pd.concat([snr_svm, snr_cnn, snr_pann], ignore_index=True)
snr_all["snr_label"] = snr_all["snr_db"].astype(str)

fig = go.Figure()
colors = {"SVM": "#636EFA", "CNN": "#EF553B", "PANN": "#00CC96"}

for name in ["SVM", "CNN", "PANN"]:
    d = snr_all[snr_all["model"] == name]
    fig.add_trace(go.Scatter(
        x=list(d["snr_label"]) + list(d["snr_label"])[::-1],
        y=list(d["f1_hi"]) + list(d["f1_lo"])[::-1],
        fill="toself", line=dict(width=0),
        fillcolor=colors[name], opacity=0.15,
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=d["snr_label"], y=d["macro_f1"], mode="lines+markers",
        name=name, line=dict(color=colors[name]),
    ))

fig.update_layout(
    title="SNR degradation curve - drone + helicopter only",
    xaxis_title="SNR (dB)", yaxis_title="Macro-F1", width=1500
)
fig.write_image("figures/snr_curve.png", scale=3)
fig.show()

SNR clean: macro_f1 = 0.9362 [0.8858, 0.9728]
SNR 10: macro_f1 = 0.8966 [0.8382, 0.9486]
SNR 5: macro_f1 = 0.8764 [0.8074, 0.9298]
SNR 0: macro_f1 = 0.7952 [0.7075, 0.8702]
SNR -5: macro_f1 = 0.7932 [0.6996, 0.8705]
SNR clean: macro_f1 = 1.0000 [1.0000, 1.0000]
SNR 10: macro_f1 = 0.9719 [0.9329, 1.0000]
SNR 5: macro_f1 = 0.9595 [0.9179, 0.9878]
SNR 0: macro_f1 = 0.8730 [0.7974, 0.9299]
SNR -5: macro_f1 = 0.8343 [0.7577, 0.9020]
SNR clean: macro_f1 = 1.0000 [1.0000, 1.0000]
SNR 10: macro_f1 = 0.9712 [0.9379, 0.9945]
SNR 5: macro_f1 = 0.9198 [0.8599, 0.9658]
SNR 0: macro_f1 = 0.8573 [0.7826, 0.9166]
SNR -5: macro_f1 = 0.8101 [0.7294, 0.8762]
